In [ ]:
from sklearn.preprocessing import StandardScaler
import pandas as pd
import mygene

In [ ]:
GEM_genes = pd.read_csv('../mouse_genes.csv') # load from mouseGEM
#print(GEM_genes)
GEM_genes_short = pd.read_csv('../mouse_genes_shortnames.csv') # load from mouseGEM
#print(GEM_genes_short)

transcriptomics = pd.read_csv('../Transcriptomics_datasets/process_trans_combinedisoforms.csv')
genes_in_transcriptomics = transcriptomics['IDFemaleAgingColony']
#print(genes_in_transcriptomics)

shared_genes = set(GEM_genes['geneID']) & set(genes_in_transcriptomics) # Find the intersection of genes in the GEM and transcriptomics datasets
not_shared_genes = set(GEM_genes['geneID']) - set(genes_in_transcriptomics) # Find the genes that are in the GEM but not in the transcriptomics dataset
print(len(shared_genes))
print(len(not_shared_genes))
print(not_shared_genes)

In [ ]:
print('Gal3st2b' in set(genes_in_transcriptomics)) # sanity check to see if a specific gene is in the transcriptomics dataset
print('Gal3st2b' in set(GEM_genes['geneID']))

In [ ]:
def find_synonyms(list_of_genes):
    """
    Find synonyms for a list of genes.
    Inputs:
        list_of_genes: list - List of gene symbols to find synonyms for
    Outputs:
        results - List of dictionaries containing gene information and synonyms
    """
    mg = mygene.MyGeneInfo()
    results = mg.querymany(list_of_genes, scopes = 'symbol,alias',
                        fields='symbol,alias,name',
                        species = 'mouse')
    return results

In [ ]:
# Look up synonyms/aliases for the genes in not_shared_genes
results = find_synonyms(not_shared_genes)

# Dictionary to store each queried gene name and its collected aliases
dictionary_with_names = {}

for result in results:
    # Original gene name used in the query
    gene_name = result.get('query')

    # Official gene symbol, if present
    official_symbol = result.get('symbol', 'N/A')

    # Alias field may be a list or a single value
    aliases = result.get('alias', [])

    # Normalize aliases so they are always stored as a list
    if not isinstance(aliases, list):
        aliases = [aliases]

    # Some gene names appear multiple times in results,
    # so append aliases if the gene is already present
    if gene_name not in dictionary_with_names:
        dictionary_with_names[gene_name] = aliases
    else:
        dictionary_with_names[gene_name].extend(aliases)

# Print all genes with their collected aliases
for name in dictionary_with_names:
    print(f"{name}: {dictionary_with_names[name]}")

# Print aliases for specific genes
print(dictionary_with_names['Nup58'])
print(dictionary_with_names['Efl1'])

In [ ]:
def replace_gene_names(list_of_genes, replacement_genes):
    # Loop through each gene in the input list by index
    for i in range(len(list_of_genes)):
        # Check each replacement gene and its list of aliases
        for replacement_gene in replacement_genes:
            # If the current gene matches one of the aliases,
            # replace it with the canonical gene name
            if list_of_genes[i] in replacement_genes[replacement_gene]:
                list_of_genes[i] = replacement_gene
                break  # Stop checking once a match is found

    # Return the updated list
    return list_of_genes

In [ ]:
# Convert genes_in_transcriptomics to a list
# (useful if it is currently a set, tuple, or another iterable)
genes_in_transcriptomics = list(genes_in_transcriptomics)

# Create a copy of the list and replace any gene aliases
# with their canonical gene names using dictionary_with_names
replaced_genes_in_transcriptomics = replace_gene_names(
    genes_in_transcriptomics.copy(),
    dictionary_with_names
)

In [ ]:
# Find genes that are present in both GEM_genes['geneID']
# and the transcriptomics gene list after replacement
shared_genes = set(GEM_genes['geneID']) & set(replaced_genes_in_transcriptomics)

# Find genes that are in GEM_genes['geneID']
# but not in the transcriptomics gene list after replacement
not_shared_genes = set(GEM_genes['geneID']) - set(replaced_genes_in_transcriptomics)

# Print how many genes are shared between the two datasets
print(len(shared_genes))

# Print how many genes are not shared
print(len(not_shared_genes))

# Print each gene that is not shared
for gene in not_shared_genes:
    print(gene)

In [ ]:
gene_dict = { # problems with names of genes
    'Atg5lrt': 'Atp5l',
    'Marchf5': 'March_5',
    'Ndufb1': '',
    'Gm3776': '',
    'Marchf8': 'March_8',
    'Il4i1': 'Il4i1b', #returned to original name because of inconsistances in naming (it already exists in mouseGEM after finding synonyms)
    # so this would be a duplicate
    'Ak9': '',
    'Gm5737': '',
    'Gal3st2': 'Gal3st2b', #returned to original name because of inconsistances in naming, same as above
    'Marchf4': 'March_4',
    'Hsd3b4': 'Hsd3b8', #returned to original name because of inconsistances in naming, same as above
    'Marchf7': 'March_7',
    'Tmlhe': '',
    'Rdh16f2': '',
    'Marchf2': 'March_2',
    'Stard9': '',
    'Marchf1': 'March_2',
    'Marchf10': 'March_10',
    'Adss2': '',
    'Sts': '',
    'Pdxp': '',
    'Marchf3': 'March_3',
    'Marchf6': 'March_6',
    'Marchf11': 'March_11',
    'Adh6b': '',
    'Ndufb4b': '',
    'Marchf9': 'March_9',
    'Faxdc2': ''
}

In [ ]:
# Loop through each key in gene_dict
for gene in gene_dict:
    # Get the gene name associated with the current key
    gene_name = gene_dict[gene]

    # Check whether this gene name appears in the GEM geneID column
    is_in_set = gene_name in set(GEM_genes["geneID"])

    # Print the result for this gene
    print(f'{gene_name} in gene set: {is_in_set}')

In [ ]:
def combine_transcriptomics_isoforms(data):
    """
    Combine transcriptomics data by averaging values for the same ID.

    Inputs:
        data: pd.DataFrame - Data with ID column and numeric features

    Outputs:
        combined_data: pd.DataFrame - Data with unique IDs and averaged values
    """
    df = data.copy()
    
    # First column is ID (e.g., 'IDFemaleAgingColony')
    id_col = df.columns[0]
    
    # Ensure numeric conversion for the rest
    numeric_data = df.iloc[:, 1:].apply(pd.to_numeric, errors='coerce')
    
    # Combine ID with numeric columns
    combined_data = pd.concat([df[[id_col]], numeric_data], axis=1)
    
    # Group by ID and average replicates
    combined_data = combined_data.groupby(id_col).mean().reset_index()
    
    # Drop columns that are all NaN (if any)
    combined_data = combined_data.dropna(axis=1, how='all')
    
    return combined_data


In [ ]:
def convert_log2rpkm_to_tpm(df, gene_col="IDFemaleAgingColony"):
    """
    Convert log2(RPKM + 1) values to TPM.

    Inputs
        df-pandas.DataFrame
            Input DataFrame with one gene/sample column and log2(RPKM + 1) expression values.
        gene_col-str
            Name of the column containing gene/sample identifiers.

    Returns
    pandas.DataFrame
        DataFrame with the same gene/sample column and TPM values.
    """

    # Separate metadata and numeric columns
    meta = df[[gene_col]]
    expr = df.drop(columns=[gene_col])

    # Convert from log2(RPKM + 1) → RPKM
    rpkm = (2 ** expr) - 1

    # Convert RPKM → TPM (normalize per column)
    tpm = rpkm.div(rpkm.sum(axis=0), axis=1) * 1e6

    # Recombine with gene/sample identifiers
    tpm_df = pd.concat([meta, tpm], axis=1)

    return tpm_df

In [ ]:
transcriptomics = pd.read_csv('../Transcriptomics_datasets/process_trans_file.csv')

# convert the transcriptomics data to TPM
transcriptomics = convert_log2rpkm_to_tpm(transcriptomics)

transcriptomics = combine_transcriptomics_isoforms(transcriptomics)

# Invert the dictionary so that each gene name maps to its key 
value_to_key = {value: key for key, values in dictionary_with_names.items() for value in values} 
print(value_to_key) 

print('March_8' in set(transcriptomics['IDFemaleAgingColony']))

# Replace names in IDFemaleAgingColony using this mapping 
transcriptomics['IDFemaleAgingColony'] = transcriptomics['IDFemaleAgingColony'].replace(value_to_key) 

# Invert the other dictionary (with manually found gene names) so that each gene name maps to its key 
value_to_key = {} 
value_to_key = {value: key for key, value in gene_dict.items() if value} 
print(value_to_key) 

# Replace names in IDFemaleAgingColony using this mapping 
transcriptomics['IDFemaleAgingColony'] = transcriptomics['IDFemaleAgingColony'].replace(value_to_key) 

print('March_8' in set(transcriptomics['IDFemaleAgingColony'])) # sanity check
print('Marchf8' in set(transcriptomics['IDFemaleAgingColony']))

genes_in_transcriptomics = transcriptomics['IDFemaleAgingColony']
shared_genes = set(GEM_genes['geneID']) & set(genes_in_transcriptomics)

# Keep only rows where IDFemaleAgingColony is in shared_genes 
filtered_transcriptomics = transcriptomics[transcriptomics['IDFemaleAgingColony'].isin(shared_genes)] 
print(filtered_transcriptomics.head()) 
print(len(filtered_transcriptomics['IDFemaleAgingColony'])) 


In [ ]:
filtered_transcriptomics.reset_index(drop=True, inplace=True) # indexing normalization
filtered_transcriptomics.to_csv('../Transcriptomics_datasets/filtered_transcriptomics_final.csv', index=False) # save file

# Proteomics

In [ ]:
# Read the full GEM gene list from a CSV file
GEM_genes = pd.read_csv('mouse_genes.csv')
# print(GEM_genes)

# Read the shortened gene name version of the GEM gene list
GEM_genes_short = pd.read_csv('mouse_genes_shortnames.csv')
# print(GEM_genes_short)

# Read the proteomics dataset
proteomics = pd.read_csv('process_prot_file.csv')

# Extract the proteomics gene/protein identifier column
genes_in_proteomics = proteomics['IDFemaleAgingColony']
# print(genes_in_transcriptomics)

# Find genes shared between the GEM gene list and the proteomics identifiers
shared_genes = set(GEM_genes['geneID']) & set(genes_in_proteomics)

# Find genes present in the GEM gene list but absent from the proteomics identifiers
not_shared_genes = set(GEM_genes['geneID']) - set(genes_in_proteomics)

# Print the number of shared genes
print(len(shared_genes))

# Print the number of genes not shared
print(len(not_shared_genes))

# Print the actual set of genes not shared
print(not_shared_genes)

In [ ]:
# Find synonyms/aliases for the genes that were not shared
results = find_synonyms(not_shared_genes)

# Dictionary to store each queried gene name and its aliases
dictionary_with_names = {}

for result in results:
    # Get the original queried gene name
    gene_name = result.get('query')

    # Get the official gene symbol, if available
    official_symbol = result.get('symbol', 'N/A')

    # Get aliases for the gene; default to an empty list if missing
    aliases = result.get('alias', [])

    # Optional debug print showing the full result for each gene
    # print(f"{gene_name}: Official={official_symbol}, Aliases={aliases}")

    # Ensure aliases is always a list
    if not isinstance(aliases, list):
        aliases = [aliases]

    # Some gene names appear multiple times in the results,
    # so either create a new entry or extend the existing alias list
    if gene_name not in dictionary_with_names:  # some gene names are duplicated
        dictionary_with_names[gene_name] = aliases
    else:
        dictionary_with_names[gene_name].extend(aliases)

# Print each gene and its collected aliases
for name in dictionary_with_names:
    print(f"{name}: {dictionary_with_names[name]}")

In [ ]:
# Convert genes_in_proteomics to a list
# This ensures it is mutable and can be processed by the replacement function
genes_in_proteomics = list(genes_in_proteomics)

# Create a copy of the proteomics gene list and replace any aliases
# with their canonical gene names using the dictionary_with_names mapping
replaced_genes_in_proteomics_mygene = replace_gene_names(
    genes_in_proteomics.copy(),
    dictionary_with_names
)

In [ ]:
# Find genes that are shared between the GEM gene list
# and the proteomics gene list after alias replacement
shared_genes_mygene = set(GEM_genes['geneID']) & set(replaced_genes_in_proteomics_mygene)

# Find genes that are still missing from the proteomics list
# even after replacing aliases with canonical gene names
not_shared_genes_mygene = set(GEM_genes['geneID']) - set(replaced_genes_in_proteomics_mygene)

# Print the number of shared genes after replacement
print(len(shared_genes_mygene))

# Print the number of genes still not shared after replacement
print(len(not_shared_genes_mygene))

In [ ]:
# to identify if the synonyms for gene names is consistent between the generic mouseGEM and databases
for gene in not_shared_genes_mygene: 
    if gene in genes_in_proteomics:
        print(gene, 'PROBLEM')

In [ ]:
from Bio import Entrez

def get_gene_synonyms(list_of_genes):
    """
    Finds official synonyms for a list of mouse gene names using the NCBI Entrez database.

    Args:
        list_of_genes (list): A list of gene symbols (strings) to query.

    Returns:
        dict: A dictionary where keys are the query genes and values are their
              corresponding lists of synonyms. If a gene is not found or has no
              synonyms, its value will be an empty list.
    """
    Entrez.email = " "  #this is necessary
    gene_synonyms_map = {}

    for gene in list_of_genes:
        synonyms = []  # Default to an empty list
        try:
            # Step 1: Search for the gene to get its unique NCBI ID
            search_handle = Entrez.esearch(
                db="gene",
                term=f"{gene}[Gene Name] AND Mus musculus[Organism]"
            )
            search_record = Entrez.read(search_handle)
            search_handle.close()

            # Proceed only if an ID was found
            if search_record["IdList"]:
                gene_id = search_record["IdList"][0]

                # Step 2: Fetch the full record for that gene ID
                fetch_handle = Entrez.efetch(db="gene", id=gene_id, retmode="xml")
                fetch_data = Entrez.read(fetch_handle)
                fetch_handle.close()

                # Step 3: Parse the fetched data to extract only the synonyms
                if fetch_data:
                    # The synonyms are located deep within the nested dictionary
                    gene_details = fetch_data[0]['Entrezgene_gene']['Gene-ref']
                    synonyms = gene_details.get('Gene-ref_syn', []) # Use .get() to safely handle missing synonyms

        except Exception as e:
            print(f"An error occurred for gene '{gene}': {e}")
        
        finally:
            # Add the result to our map
            gene_synonyms_map[gene] = synonyms
            # Be polite to NCBI servers by adding a small delay
            #time.sleep(0.4)

    return gene_synonyms_map

GEM_genes = pd.read_csv('mouse_genes.csv')
GEM_genes_short = pd.read_csv('mouse_genes_shortnames.csv')

proteomics = pd.read_csv('process_prot_file.csv')
genes_in_proteomics = proteomics['IDFemaleAgingColony']

shared_genes = set(GEM_genes['geneID']) & set(genes_in_proteomics)
not_shared_genes = set(GEM_genes['geneID']) - set(genes_in_proteomics)

my_genes = not_shared_genes
# Get the synonyms
synonym_results = get_gene_synonyms(my_genes)

# Print the results
for gene, syn_list in synonym_results.items():
    if syn_list:
        print(f"Synonyms for '{gene}': {', '.join(syn_list)}")
    else:
        print(f"No synonyms found for '{gene}'.")

In [ ]:
print(synonym_results)

In [ ]:
# Convert genes_in_proteomics to a list
# This makes sure the data is in a mutable list format
genes_in_proteomics = list(genes_in_proteomics)

# Create a copy of the proteomics gene list and replace any alias names
# using the synonym_results dictionary
replaced_genes_in_proteomics_NCBI = replace_gene_names(
    genes_in_proteomics.copy(),
    synonym_results
)

In [ ]:
# Find genes that are shared between the GEM gene list
# and the proteomics gene list after NCBI synonym replacement
shared_genes_NCBI = set(GEM_genes['geneID']) & set(replaced_genes_in_proteomics_NCBI)

# Find genes that are still missing from the proteomics list
# after replacing aliases using NCBI synonym data
not_shared_genes_NCBI = set(GEM_genes['geneID']) - set(replaced_genes_in_proteomics_NCBI)

# Print the number of shared genes after NCBI-based replacement
print(len(shared_genes_NCBI))

# Print the number of genes still not shared after NCBI-based replacement
print(len(not_shared_genes_NCBI))

# Print each gene that is still not shared
for gene in not_shared_genes_NCBI:
    print(gene)

In [ ]:
# Combine the gene names obtained after NCBI-based replacement
# and MyGene-based replacement into one set
combined_genes = set(replaced_genes_in_proteomics_NCBI) | set(replaced_genes_in_proteomics_mygene)

# Find genes from the GEM gene list that are present
# in the combined proteomics gene set
shared_genes = set(GEM_genes['geneID']) & combined_genes

# Find genes from the GEM gene list that are still missing
# after combining both replacement approaches
not_shared_genes = set(GEM_genes['geneID']) - shared_genes

# Print the number of shared genes
print(len(shared_genes))

# Print the number of genes still not shared
print(len(not_shared_genes))

In [ ]:
# Create a dictionary to store genes with inconsistent synonym mappings
# between mouseGEM and the external databases
dictionary_of_inconsistent_genes = {}

# Loop through genes that are still not shared
for gene in not_shared_genes:
    # Check whether this gene actually appears in the original proteomics list
    # If it does, this suggests it may have been incorrectly treated as a synonym
    if gene in genes_in_proteomics:
        print(gene, 'PROBLEM - recognized as synonym while it should not be')

        # Search through the synonym dictionary to find which canonical gene
        # incorrectly included this gene as one of its synonyms
        for synonym in synonym_results:
            if gene in synonym_results[synonym]:
                print(synonym)

                # Store the problematic mapping:
                # key = inconsistent gene
                # value = canonical gene it was matched to
                dictionary_of_inconsistent_genes[gene] = synonym

In [ ]:
print(dictionary_of_inconsistent_genes)

In [ ]:
proteomics = pd.read_csv('process_prot_file.csv') 

# Invert the dictionary so that each gene name maps to its key using the mygene data
value_to_key = {value: key for key, values in dictionary_with_names.items() for value in values} 
print(value_to_key) 

# Replace names in IDFemaleAgingColony using this mapping 
proteomics['IDFemaleAgingColony'] = proteomics['IDFemaleAgingColony'].replace(value_to_key) 

# Replace names in IDFemaleAgingColony using the mapping from the dictionary with inconsistent gene names
proteomics['IDFemaleAgingColony'] = proteomics['IDFemaleAgingColony'].replace(dictionary_of_inconsistent_genes) 

genes_in_proteomics = proteomics['IDFemaleAgingColony']
shared_genes = set(GEM_genes['geneID']) & set(genes_in_proteomics)

# Keep only rows where IDFemaleAgingColony is in shared_genes 
filtered_proteomics = proteomics[proteomics['IDFemaleAgingColony'].isin(shared_genes)] 
print(proteomics.head()) 
print(len(filtered_proteomics['IDFemaleAgingColony'])) 

In [ ]:
# save files
filtered_proteomics.reset_index(drop=True, inplace=True)
filtered_proteomics.to_csv('../Proteomics_datasets/filtered_proteomics_final.csv', index=False)